In [4]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.6-luna")
model = init_chat_model(MODEL_ID)

In [5]:
from langchain.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

SYSTEM_PROMPT = "당신은 친절하고 정확한 은행 상담 챗봇입니다."

def call_model(state: MessagesState):
    response = model.invoke(
        [SystemMessage(content=SYSTEM_PROMPT), *state["messages"]]
    )

    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")
builder.add_edge("model", END)

chat = builder.compile(checkpointer=InMemorySaver())

In [6]:
config = {"configurable": {"thread_id": "bank-account-demo"}}

def ask(text: str) -> str:
    result = chat.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config=config,
    )
    return result["messages"][-1].content

In [7]:
print(
    ask(
        "안녕하세요. 비대면으로 은행 계좌를 개설하고 싶습니다. "
        "어떻게 시작해야 하나요?"
    )
)

안녕하세요! 비대면 계좌 개설은 보통 다음 순서로 진행합니다.

1. **은행 공식 앱 설치**
   - 이용하려는 은행의 공식 모바일 앱을 설치하세요.
   - 앱스토어·구글플레이에서 개발자명이 은행명인지 확인하고, 문자나 검색광고 링크보다는 공식 경로를 이용하는 것이 안전합니다.

2. **계좌 개설 메뉴 선택**
   - 앱에서 **“입출금 계좌 개설”**, **“비대면 계좌 개설”** 등의 메뉴를 선택합니다.

3. **본인 확인**
   - 휴대전화 본인인증
   - 신분증 촬영 또는 모바일 신분증 인증
   - 얼굴 촬영·영상통화 등 추가 인증  
   이 진행될 수 있습니다.

4. **고객정보 및 거래 목적 입력**
   - 주소, 직업, 소득·거래 목적, 자금 원천 등을 입력합니다.
   - 금융거래 목적을 정확하게 선택해야 합니다.

5. **약관 동의 및 계좌 비밀번호 설정**
   - 상품 설명과 약관을 확인한 뒤 비밀번호를 설정합니다.

6. **추가 인증 진행**
   - 기존 은행 계좌로 1원 인증을 하거나, 영상·상담원 인증을 요구할 수 있습니다.

7. **개설 완료**
   - 심사가 완료되면 계좌번호가 발급됩니다.
   - 체크카드, 인터넷뱅킹, 이체 한도 등의 서비스는 별도로 신청할 수 있습니다.

**미리 준비할 것**
- 본인 명의 휴대전화
- 주민등록증 또는 운전면허증 등 신분증
- 본인 명의의 기존 은행 계좌(필요한 경우)
- 안정적인 인터넷 연결과 밝은 촬영 환경

참고로 처음 개설한 계좌는 금융사기 예방을 위해 **이체 한도가 제한된 한도제한계좌**로 개설될 수 있으며, 이후 거래 목적을 증빙하면 한도 해제가 가능한 경우가 많습니다. 또한 만 14세 미만, 외국인, 법인 등은 절차와 필요 서류가 다를 수 있습니다.

어느 은행의 계좌를 개설하려는지 알려주시면 해당 은행 앱 기준으로 더 구체적으로 안내해 드리겠습니다.


In [8]:
print(ask("방금 안내한 절차를 불렛포인트로 다시 정리해 주세요."))

- **은행 공식 앱 설치**
  - 앱스토어 또는 구글플레이에서 은행 공식 앱을 설치합니다.
  - 개발자명이 은행명인지 확인하고, 공식 경로를 이용하세요.

- **비대면 계좌 개설 메뉴 선택**
  - 앱에서 **‘입출금 계좌 개설’** 또는 **‘비대면 계좌 개설’**을 선택합니다.

- **본인 확인**
  - 휴대전화 본인인증
  - 신분증 촬영 또는 모바일 신분증 인증
  - 필요 시 얼굴 촬영·영상통화 인증

- **고객정보 및 거래 목적 입력**
  - 주소, 직업, 소득, 거래 목적, 자금 원천 등을 정확히 입력합니다.

- **약관 동의 및 계좌 비밀번호 설정**
  - 상품 설명과 약관을 확인한 뒤 계좌 비밀번호를 설정합니다.

- **추가 인증**
  - 본인 명의의 기존 은행 계좌를 통한 1원 인증이나 영상·상담원 인증을 진행할 수 있습니다.

- **계좌 개설 완료**
  - 심사가 완료되면 계좌번호가 발급됩니다.
  - 체크카드, 인터넷뱅킹, 이체 한도 등은 별도로 신청할 수 있습니다.

- **사전에 준비할 것**
  - 본인 명의 휴대전화
  - 주민등록증 또는 운전면허증 등 신분증
  - 본인 명의의 기존 은행 계좌
  - 안정적인 인터넷 연결과 밝은 촬영 환경

- **참고사항**
  - 금융사기 예방을 위해 처음에는 **한도제한계좌**로 개설될 수 있습니다.
  - 거래 목적을 증빙하면 이체 한도 해제가 가능한 경우가 많습니다.
  - 미성년자, 외국인, 법인은 절차와 필요 서류가 다를 수 있습니다.


In [9]:
snapshot = chat.get_state(config)

for message in snapshot.values["messages"]:
    print(f"[{message.type}] {message.content}\n")

[human] 안녕하세요. 비대면으로 은행 계좌를 개설하고 싶습니다. 어떻게 시작해야 하나요?

[ai] 안녕하세요! 비대면 계좌 개설은 보통 다음 순서로 진행합니다.

1. **은행 공식 앱 설치**
   - 이용하려는 은행의 공식 모바일 앱을 설치하세요.
   - 앱스토어·구글플레이에서 개발자명이 은행명인지 확인하고, 문자나 검색광고 링크보다는 공식 경로를 이용하는 것이 안전합니다.

2. **계좌 개설 메뉴 선택**
   - 앱에서 **“입출금 계좌 개설”**, **“비대면 계좌 개설”** 등의 메뉴를 선택합니다.

3. **본인 확인**
   - 휴대전화 본인인증
   - 신분증 촬영 또는 모바일 신분증 인증
   - 얼굴 촬영·영상통화 등 추가 인증  
   이 진행될 수 있습니다.

4. **고객정보 및 거래 목적 입력**
   - 주소, 직업, 소득·거래 목적, 자금 원천 등을 입력합니다.
   - 금융거래 목적을 정확하게 선택해야 합니다.

5. **약관 동의 및 계좌 비밀번호 설정**
   - 상품 설명과 약관을 확인한 뒤 비밀번호를 설정합니다.

6. **추가 인증 진행**
   - 기존 은행 계좌로 1원 인증을 하거나, 영상·상담원 인증을 요구할 수 있습니다.

7. **개설 완료**
   - 심사가 완료되면 계좌번호가 발급됩니다.
   - 체크카드, 인터넷뱅킹, 이체 한도 등의 서비스는 별도로 신청할 수 있습니다.

**미리 준비할 것**
- 본인 명의 휴대전화
- 주민등록증 또는 운전면허증 등 신분증
- 본인 명의의 기존 은행 계좌(필요한 경우)
- 안정적인 인터넷 연결과 밝은 촬영 환경

참고로 처음 개설한 계좌는 금융사기 예방을 위해 **이체 한도가 제한된 한도제한계좌**로 개설될 수 있으며, 이후 거래 목적을 증빙하면 한도 해제가 가능한 경우가 많습니다. 또한 만 14세 미만, 외국인, 법인 등은 절차와 필요 서류가 다를 수 있습니다.

어느 은행의 계좌를 개설하려는지 알려주시면 해당 은행 앱 기준으로 더 구체적으로 안내해 드리겠습

In [10]:
other_config = {"configurable": {"thread_id": "another-customer"}}
result = chat.invoke(
    {"messages": [{"role": "user", "content": "제가 방금 무엇을 물었나요?"}]},
    config=other_config,
)
print(result["messages"][-1].content)

방금 “제가 방금 무엇을 물었나요?”라고 물으셨습니다.
